# K-Modes Clustering on Retail Customer Data

**Objective:** Cluster retail customers using **categorical** attributes (region, product category, channel) with K-Modes.

**Why K-Modes?** Standard K-Means cannot handle categorical variables directly. K-Modes uses mode (most frequent category) as the cluster centroid and dissimilarity based on matching categories.

**Pipeline:** Load Data → EDA → Encode Categories → Cost Function → K-Modes → Cluster Profiling

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from kmodes.kmodes import KModes

from retail_data import generate_retail_dataset

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_STATE = 42

## 1. Load Retail Dataset

In [ ]:
df = generate_retail_dataset(n_samples=2000, random_state=RANDOM_STATE)
print(f'Shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis (Categorical Features)

In [ ]:
cat_cols = ['Region', 'Product_Category', 'Purchase_Channel', 'Customer_Segment']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, col in zip(axes.ravel(), cat_cols):
    counts = df[col].value_counts()
    sns.barplot(x=counts.index, y=counts.values, ax=ax, palette='viridis')
    ax.set_title(f'Distribution of {col}')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
print('Unique values per categorical column:')
for col in cat_cols:
    print(f'  {col}: {df[col].nunique()} -> {df[col].unique().tolist()}')

## 3. Prepare Categorical Features for K-Modes

K-Modes requires categorical data as strings or integers. We use shopping-behavior categories only (exclude target segment for unsupervised clustering).

In [ ]:
feature_cols = ['Region', 'Product_Category', 'Purchase_Channel']
X_cat = df[feature_cols].astype(str).values
print(f'Categorical feature matrix shape: {X_cat.shape}')
X_cat[:5]

## 4. Find Optimal K (Cost Function)

In [ ]:
k_range = range(2, 9)
costs = []

for k in k_range:
    km = KModes(n_clusters=k, init='Huang', n_init=5, random_state=RANDOM_STATE)
    km.fit(X_cat)
    costs.append(km.cost_)
    print(f'K={k}, Cost={km.cost_:.2f}')

plt.figure(figsize=(10, 5))
plt.plot(k_range, costs, 'ro-')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Cost (Total Dissimilarity)')
plt.title('K-Modes Cost Function')
plt.show()

## 5. Train Final K-Modes Model

In [ ]:
OPTIMAL_K = 4
kmodes = KModes(n_clusters=OPTIMAL_K, init='Huang', n_init=10, random_state=RANDOM_STATE)
df['Cluster'] = kmodes.fit_predict(X_cat)

print('Cluster distribution:')
print(df['Cluster'].value_counts().sort_index())
print(f'\nFinal Cost: {kmodes.cost_:.2f}')
print(f'\nCluster Centroids (modes):')
centroids_df = pd.DataFrame(kmodes.cluster_centroids_, columns=feature_cols)
centroids_df.index.name = 'Cluster'
centroids_df

## 6. Visualize Cluster Composition

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, feature_cols):
    ct = pd.crosstab(df['Cluster'], df[col], normalize='index')
    ct.plot(kind='bar', stacked=True, ax=ax, colormap='Set3')
    ax.set_title(f'{col} by Cluster')
    ax.set_xlabel('Cluster')
    ax.legend(title=col, bbox_to_anchor=(1.02, 1))
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='Cluster', y='Total_Sales', palette='Set2')
plt.title('Total Sales Distribution per K-Modes Cluster')
plt.show()

## 7. Cluster Profiling

In [ ]:
for cluster_id in sorted(df['Cluster'].unique()):
    subset = df[df['Cluster'] == cluster_id]
    print(f'\n--- Cluster {cluster_id} (n={len(subset)}) ---')
    for col in feature_cols:
        top = subset[col].mode()[0]
        pct = (subset[col] == top).mean() * 100
        print(f'  {col}: {top} ({pct:.1f}%)')
    print(f'  Avg Total Sales: ${subset["Total_Sales"].mean():,.2f}')
    print(f'  Avg Annual Income: ${subset["Annual_Income"].mean():,.0f}')

In [ ]:
pd.crosstab(df['Cluster'], df['Customer_Segment'], normalize='index').round(3)

## 8. Conclusions

- **K-Modes** is ideal when retail segmentation is driven by categorical preferences (region, category, channel).
- Each cluster centroid represents the most common category combination — easy to interpret for marketing teams.
- Cross-reference with `Total_Sales` to prioritize high-value categorical segments.
- **Limitation:** K-Modes ignores numerical features; use K-Prototypes when you need both categorical and numerical data.